In [0]:
-- Crear el schema gold dentro del catálogo
CREATE SCHEMA IF NOT EXISTS dev_workspace.gold;

-- =============================================================
-- dim_genre
-- =============================================================
-- One row per unique genre.
-- Surrogate key assigned during Silver → Gold transformation.


CREATE TABLE IF NOT EXISTS dev_workspace.gold.dim_genre (
    genre_id    INT         NOT NULL,
    genre_name  STRING      NOT NULL
)
USING DELTA
COMMENT 'Genre dimension. One row per unique genre from silver.spotify_tracks.';


-- =============================================================
-- dim_artist
-- =============================================================
-- One row per unique artist.
-- Sourced by exploding the artists ARRAY<STRING> from Silver.

CREATE TABLE IF NOT EXISTS dev_workspace.gold.dim_artist (
    artist_id   INT         NOT NULL,
    artist_name STRING      NOT NULL
)
USING DELTA
COMMENT 'Artist dimension. One row per unique artist, exploded from silver.spotify_tracks.artists array.';


-- =============================================================
-- dim_album
-- =============================================================
-- One row per unique album.
-- Links to dim_artist via artist_id (snowflake normalization).

CREATE TABLE IF NOT EXISTS dev_workspace.gold.dim_album (
    album_id    INT         NOT NULL,
    album_name  STRING      NOT NULL,
    artist_id   INT                     -- FK → dim_artist (main artist of the album)
)
USING DELTA
COMMENT 'Album dimension. Links to dim_artist via artist_id (snowflake normalization).';


ALTER TABLE dev_workspace.gold.dim_album
    ADD CONSTRAINT chk_album_artist_id
    CHECK (artist_id IS NULL OR artist_id > 0);


-- =============================================================
-- dim_audio_features
-- =============================================================
-- One row per track containing all Spotify audio analysis columns.
-- Separated from fact_tracks to normalize the snowflake schema.

CREATE TABLE IF NOT EXISTS dev_workspace.gold.dim_audio_features (
    audio_id            INT         NOT NULL,
    track_id            STRING      NOT NULL,   -- natural key back to the source track
    danceability        DOUBLE,                 -- 0.0 – 1.0
    energy              DOUBLE,                 -- 0.0 – 1.0
    key                 STRING,                 -- musical note label (C, C#, D, ...)
    loudness            DOUBLE,                 -- dB, typically -60 to 0
    mode                STRING,                 -- Major / Minor
    speechiness         DOUBLE,                 -- 0.0 – 1.0
    acousticness        DOUBLE,                 -- 0.0 – 1.0
    instrumentalness    DOUBLE,                 -- 0.0 – 1.0
    liveness            DOUBLE,                 -- 0.0 – 1.0
    valence             DOUBLE,                 -- 0.0 – 1.0
    tempo               DOUBLE,                 -- BPM
    time_signature      INT                     -- beats per bar
)
USING DELTA
COMMENT 'Audio features dimension. One row per track_id. Snowflake normalization of audio analysis columns.';


ALTER TABLE dev_workspace.gold.dim_audio_features
    ADD CONSTRAINT chk_af_danceability
    CHECK (danceability IS NULL OR (danceability >= 0.0 AND danceability <= 1.0));

ALTER TABLE dev_workspace.gold.dim_audio_features
    ADD CONSTRAINT chk_af_energy
    CHECK (energy IS NULL OR (energy >= 0.0 AND energy <= 1.0));

ALTER TABLE dev_workspace.gold.dim_audio_features
    ADD CONSTRAINT chk_af_speechiness
    CHECK (speechiness IS NULL OR (speechiness >= 0.0 AND speechiness <= 1.0));

ALTER TABLE dev_workspace.gold.dim_audio_features
    ADD CONSTRAINT chk_af_acousticness
    CHECK (acousticness IS NULL OR (acousticness >= 0.0 AND acousticness <= 1.0));

ALTER TABLE dev_workspace.gold.dim_audio_features
    ADD CONSTRAINT chk_af_instrumentalness
    CHECK (instrumentalness IS NULL OR (instrumentalness >= 0.0 AND instrumentalness <= 1.0));

ALTER TABLE dev_workspace.gold.dim_audio_features
    ADD CONSTRAINT chk_af_liveness
    CHECK (liveness IS NULL OR (liveness >= 0.0 AND liveness <= 1.0));

ALTER TABLE dev_workspace.gold.dim_audio_features
    ADD CONSTRAINT chk_af_valence
    CHECK (valence IS NULL OR (valence >= 0.0 AND valence <= 1.0));

ALTER TABLE dev_workspace.gold.dim_audio_features
    ADD CONSTRAINT chk_af_tempo
    CHECK (tempo IS NULL OR tempo > 0);


-- =============================================================
-- fact_tracks
-- =============================================================
-- Central fact table. One row per (track, artist) combination.
-- Holds all surrogate FK references and the three measures.
-- Must be created after all dimension tables exist.

CREATE TABLE IF NOT EXISTS dev_workspace.gold.fact_tracks (
    fact_id             BIGINT      NOT NULL,
    track_id            STRING      NOT NULL,
    track_name          STRING      NOT NULL,
    artist_id           INT                 ,   -- FK → dim_artist
    album_id            INT                 ,   -- FK → dim_album
    genre_id            INT                 ,   -- FK → dim_genre
    audio_id            INT                 ,   -- FK → dim_audio_features
    popularity          INT                 ,   -- measure: 0 – 100
    duration_ms         BIGINT              ,   -- measure: track length in milliseconds
    duration_formatted  STRING, 
    explicit            BOOLEAN                 -- measure: explicit content flag
)
USING DELTA
COMMENT 'Fact table. One row per (track, artist). FKs reference all four dimension tables.'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
);

ALTER TABLE dev_workspace.gold.fact_tracks
    ADD CONSTRAINT chk_fact_popularity
    CHECK (popularity IS NULL OR (popularity >= 0 AND popularity <= 100));

ALTER TABLE dev_workspace.gold.fact_tracks
    ADD CONSTRAINT chk_fact_duration
    CHECK (duration_ms IS NULL OR duration_ms > 0);

ALTER TABLE dev_workspace.gold.fact_tracks
    ADD CONSTRAINT chk_fact_artist_id
    CHECK (artist_id IS NULL OR artist_id > 0);

ALTER TABLE dev_workspace.gold.fact_tracks
    ADD CONSTRAINT chk_fact_album_id
    CHECK (album_id IS NULL OR album_id > 0);

ALTER TABLE dev_workspace.gold.fact_tracks
    ADD CONSTRAINT chk_fact_genre_id
    CHECK (genre_id IS NULL OR genre_id > 0);

ALTER TABLE dev_workspace.gold.fact_tracks
    ADD CONSTRAINT chk_fact_audio_id
    CHECK (audio_id IS NULL OR audio_id > 0);